# Open-source lakehouse demo
Spark + Delta Lake on SeaweedFS (S3). This is the same PySpark you would run in a Databricks notebook.

In [1]:
from lakehouse.common import spark, table_path
from delta.tables import DeltaTable
ss = spark()
print(ss.version)

4.2.0


## 1. Medallion layers: Bronze → Silver → Gold

In [2]:
from lakehouse.publish import delta_tables
for layer in ['bronze', 'silver', 'gold']:
    print(layer.upper(), delta_tables(layer))

BRONZE ['document_files', 'erp_asset', 'erp_employee', 'erp_plot', 'erp_purchase_order', 'erp_tenant', 'erp_vendor', 'iot_bus', 'photo_files', 'ticketing', 'weighbridge']
SILVER ['asset', 'bus_telemetry', 'document', 'employee', 'photo', 'plot', 'purchase_order', 'tenant', 'ticket', 'vendor', 'weighbridge']
GOLD ['asset_condition', 'asset_photos', 'bus_daily', 'document_catalog', 'estate_summary', 'procurement_monthly', 'tenant_360', 'tenant_tickets', 'weighbridge_daily']


In [3]:
ss.read.format('delta').load(table_path('gold', 'tenant_360'))\
  .select('tenant_id', 'tenant_name', 'area_ha', 'contract_value_idr', 'open_tickets', 'tonnage_30d').show(truncate=False)

+---------+---------------------+-------+------------------+------------+-----------+
|tenant_id|tenant_name          |area_ha|contract_value_idr|open_tickets|tonnage_30d|
+---------+---------------------+-------+------------------+------------+-----------+
|T004     |Steelpipe Utama      |13.62  |14859000000       |6           |1554.3     |
|T006     |Northline Logistics  |15.55  |17167000000       |3           |1592.4     |
|T008     |Stride Footwear      |5.58   |5730000000        |1           |1614.5     |
|T009     |Ciptadaya Electronics|4.13   |4138000000        |3           |1543.4     |
|T001     |Prima Glass Works    |8.33   |7788000000        |3           |1577.6     |
|T005     |Pesisir Chemicals    |16.95  |17339000000       |3           |1627.4     |
|T007     |Solaris Panel        |20.53  |20550000000       |3           |1615.2     |
|T010     |Lestari Foods        |15.89  |17908000000       |1           |1327.2     |
|T002     |GreenCell Battery    |17.56  |17437000000  

## 2. Unstructured data: scanned PDF → text (OCR) → structured fields

In [4]:
doc = ss.read.format('delta').load(table_path('silver', 'document'))
doc.select('file_name', 'extraction_method', 'doc_type', 'doc_number', 'tenant_id', 'amount_idr', 'doc_date')\
   .orderBy('extraction_method').show(30, truncate=False)

+---------------------+-----------------+-----------------+----------------+---------+------------+----------+
|file_name            |extraction_method|doc_type         |doc_number      |tenant_id|amount_idr  |doc_date  |
+---------------------+-----------------+-----------------+----------------+---------+------------+----------+
|HANDOVER_T001_001.pdf|OCR_TESSERACT    |PLOT HANDOVER    |HOC/EST/2026/001|T001     |NULL        |2026-03-18|
|HANDOVER_T005_005.pdf|OCR_TESSERACT    |PLOT HANDOVER    |HOC/EST/2026/005|T005     |NULL        |2026-08-28|
|HANDOVER_T003_003.pdf|OCR_TESSERACT    |PLOT HANDOVER    |HOC/EST/2026/003|T003     |NULL        |2026-03-21|
|HANDOVER_T007_007.pdf|OCR_TESSERACT    |PLOT HANDOVER    |HOC/EST/2026/007|T007     |NULL        |2026-06-18|
|PERMIT_T002.pdf      |OCR_TESSERACT    |BUILDING APPROVAL|BPA/EST/2026/002|T002     |684000000000|2026-02-13|
|PERMIT_T006.pdf      |OCR_TESSERACT    |BUILDING APPROVAL|BPA/EST/2026/006|T006     |116000000000|2026-06-22|
|

In [5]:
print(doc.filter("extraction_method = 'OCR_TESSERACT'").select('text').first()[0])

BUILDING APPROVAL

Number: BPA/EST/2026/006

Date: 22 June 2026

Applicant: Northline Logistics

Tenant Code: TOO6

Location: Plot P-A06

Building use: Logistics & Warehousing factory
Investment Value: IDR 116,000,000,000

This approval is valid while the building is used as approved.



## 3. Photos: EXIF metadata (timestamp, GPS, asset code)

In [6]:
ss.read.format('delta').load(table_path('gold', 'asset_photos'))\
  .select('file_name', 'asset_id', 'asset_name', 'taken_at', 'lat', 'lon').show(5, truncate=False)

+-----------------------------+--------+------------------------+-------------------+------------------+------------------+
|file_name                    |asset_id|asset_name              |taken_at           |lat               |lon               |
+-----------------------------+--------+------------------------+-------------------+------------------+------------------+
|PHOTO_20260906_085106_451.jpg|AST-0008|Water Treatment Plant 08|2026-09-06 08:51:06|-6.88378611111111 |107.61384444444444|
|PHOTO_20261012_050306_826.jpg|AST-0008|Water Treatment Plant 08|2026-10-12 05:03:06|-6.88378611111111 |107.61384444444444|
|PHOTO_20260923_120106_136.jpg|AST-0006|Water Treatment Plant 06|2026-09-23 12:01:06|-6.884411111111111|107.60969722222222|
|PHOTO_20260907_045306_756.jpg|AST-0007|Water Treatment Plant 07|2026-09-07 04:53:06|-6.911575         |107.59371388888889|
|PHOTO_20261016_125806_264.jpg|AST-0012|Wastewater Plant 12     |2026-10-16 12:58:06|-6.896699999999999|107.58592499999999|
+-------

## 4. CDC & time travel: change history of a Silver table

In [7]:
DeltaTable.forPath(ss, table_path('silver', 'purchase_order')).history()\
  .select('version', 'timestamp', 'operation', 'operationMetrics.numTargetRowsInserted', 'operationMetrics.numTargetRowsUpdated')\
  .show(truncate=False)

+-------+-------------------+---------+---------------------+--------------------+
|version|timestamp          |operation|numTargetRowsInserted|numTargetRowsUpdated|
+-------+-------------------+---------+---------------------+--------------------+
|0      |2026-10-27 13:28:21|WRITE    |NULL                 |NULL                |
+-------+-------------------+---------+---------------------+--------------------+



In [8]:
v0 = ss.read.format('delta').option('versionAsOf', 0).load(table_path('silver', 'purchase_order'))
now = ss.read.format('delta').load(table_path('silver', 'purchase_order'))
print('POs at version 0:', v0.count())
print('POs now         :', now.count())
now.groupBy('status').count().join(v0.groupBy('status').count().withColumnRenamed('count', 'count_v0'), 'status').show()

POs at version 0: 600


POs now         : 600


+---------+-----+--------+
|   status|count|count_v0|
+---------+-----+--------+
|    DRAFT|   42|      42|
|     PAID|  304|     304|
| APPROVED|  102|     102|
|DELIVERED|  152|     152|
+---------+-----+--------+

